# ETL Pipeline – Slutuppgift Data Science

Denna notebook implementerar en ETL-pipeline som körs på två dataset:
- main dataset
- validation dataset


In [1]:
!pip install pandas numpy python-dotenv matplotlib seaborn langchain langchain_groq

  Using cached numpy-2.4.1-cp314-cp314-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-macosx_10_13_x86_64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached langchain-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_groq-1.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp314-cp314-macosx_10_15_x86_64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp314-cp314-macosx_10_13_x86_64.whl.metadata (6.3 kB)
  Using cached langchain_core-1.2.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph-1.0.6-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-no

# IMPORTERA BIBLOTEK OCH LLM

Här kommer alla importer jag behöver för att kunna slutföra uppgiften.
Även här så kommer jag att ha min LLM.

In [2]:
import os
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Saknar GROQ_API_KEY i .env"


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

if llm:
    print("LLM is initialized")

/Users/rikardsoderstrom/DataScience-uppgift/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


LLM is initialized


# Ladda in och inspektera dataset


In [10]:
df_raw = pd.read_csv("nordtech_data.csv")

df_raw.head()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
0,ORD-2024-00001,ORD-2024-00001-1,2024-05-19,2024-05-22,SKU-WC001,Webbkamera HD,Tillbehör,1,SEK 799,Uppsala,Privat,Kort,KND-53648,Levererad,NaN,NaN,NaN
1,ORD-2024-00002,ORD-2024-00002-1,2024-12-02,5 december 2024,SKU-HB001,USB-C Hub 7-port,Tillbehör,1,549.00,Göteborg,Privat,Swish,KND-84095,Levererad,NaN,NaN,NaN
2,ORD-2024-00003,ORD-2024-00003-1,2024-12-31,2025-01-03,SKU-SD001,Extern SSD 1TB,Lagring,1,1199.00,NaN,Företag,Faktura,KND-91748,Levererad,Stämmer inte överens med produktbeskrivningen.,2025-01-12,2.0
3,ORD-2024-00003,ORD-2024-00003-2,2024-12-31,2025-01-03,SKU-SD002,Extern SSD 500GB,Lagring,10,699 kr,Stockholm,Företag,FAKTURA,KND-91748,Mottagen,"Leveransen tog lite längre än utlovat, men pro...",2025-01-14,3.0
4,ORD-2024-00003,ORD-2024-00003-3,2024-12-31,2025-01-03,SKU-MS001,Trådlös Mus X1,Tillbehör,1,399.00,Stockholm,Företag,Faktura,KND-91748,NaN,NaN,NaN,NaN


In [11]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         2767 non-null   str    
 1   orderrad_id      2767 non-null   str    
 2   orderdatum       2767 non-null   str    
 3   leveransdatum    2767 non-null   str    
 4   produkt_sku      2767 non-null   str    
 5   produktnamn      2767 non-null   str    
 6   kategori         2767 non-null   str    
 7   antal            2767 non-null   str    
 8   pris_per_enhet   2767 non-null   str    
 9   region           2612 non-null   str    
 10  kundtyp          2767 non-null   str    
 11  betalmetod       2651 non-null   str    
 12  kund_id          2767 non-null   str    
 13  leveransstatus   2673 non-null   str    
 14  recension_text   1355 non-null   str    
 15  recensionsdatum  1355 non-null   str    
 16  betyg            1355 non-null   float64
dtypes: float64(1), str(16)
me

In [12]:
df_raw.tail()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2762,ORD-2024-01654,ORD-2024-01654-3,2024-10-04,2024-10-07,SKU-LP003,Laptop Gaming X,Datorer,2,18999.00,stockholm,Företag,NaN,KND-99742,Levererad,"Förväntade mig mer för priset, men den duger.",2024-10-08,3.0
2763,ORD-2024-01655,ORD-2024-01655-1,2024-01-15,2024-01-17,SKU-MN003,"Bildskärm 32"" Curved",Bildskärmar,1,5999.00,Göteborg,Privat,Faktura,KND-83827,Returnerad,Stämmer inte överens med produktbeskrivningen.,2024-01-27,2.0
2764,ORD-2024-01656,ORD-2024-01656-1,2024-07-29,2024-07-31,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,göteborg,Privat,faktura,KND-60471,Levererad,NaN,NaN,NaN
2765,ORD-2024-01656,ORD-2024-01656-2,2024-07-29,2024-07-31,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Göteborg,Konsument,Faktura,KND-60471,Levererad,NaN,NaN,NaN
2766,ORD-2024-01657,ORD-2024-01657-1,2024-07-03,2024-07-05,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,STOCKHOLM,Privat,NaN,KND-26325,levererad,Överträffade mina förväntningar. 5 av 5!,2024-07-08,4.0


In [13]:
df_raw.sample(10)

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2010,ORD-2024-01205,ORD-2024-01205-1,2024-05-06,2024-05-08,SKU-HS001,Headset Pro ANC,Ljud,2,SEK 1899,Stockholm,Privat,Kort,KND-41689,Levererad,NaN,NaN,NaN
855,ORD-2024-00531,ORD-2024-00531-2,2024-07-16,2024-07-19,SKU-MN003,"Bildskärm 32"" Curved",Bildskärmar,1,5999 kr,UPPSALA,Privat,Swish,KND-61712,Levererad,"Leverans på två dagar, imponerad!",28 juli 2024,5.0
2746,ORD-2024-01646,ORD-2024-01646-2,"February 01, 2024",2024-02-04,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Stockholm,Privat,Faktura,KND-92109,Levererad,Medelmåttig upplevelse.,8 februari 2024,3.0
604,ORD-2024-00375,ORD-2024-00375-2,2024-01-05,2024-01-08,SKU-LP001,Laptop Pro 15,Datorer,1,14999.00,Stockholm,Privat,Swish,KND-44657,Levererad,NaN,NaN,NaN
2372,ORD-2024-01418,ORD-2024-01418-2,2024-07-02,2024-07-05,SKU-HS002,Headset Budget,Ljud,1,499.00,göteborg,PRIVAT,Kort,KND-27754,Levererad,NaN,NaN,NaN
1926,ORD-2024-01159,ORD-2024-01159-1,14 december 2024,2024-12-17,SKU-SD002,Extern SSD 500GB,Lagring,1,699.00,ÖREBRO,Privat,Kort,KND-69711,Levererad,Toppen! Exakt vad jag behövde.,2024-12-21,4.0
789,ORD-2024-00489,ORD-2024-00489-2,2024-02-19,2024-02-22,SKU-HS002,Headset Budget,Ljud,1,499.00,Stockholm,Företag,Faktura,KND-83717,Levererad,NaN,NaN,NaN
2585,ORD-2024-01547,ORD-2024-01547-1,2024-12-08,2024-12-11,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Örebro,PRIVAT,Faktura,KND-81067,Levererad,NaN,NaN,NaN
402,ORD-2024-00250,ORD-2024-00250-1,2024-11-20,2024-11-17,SKU-KB001,Mekaniskt Tangentbord K7,Tillbehör,1,1299.00,Malmö,privat,Faktura,KND-56094,Levererad,NaN,NaN,NaN
1938,ORD-2024-01166,ORD-2024-01166-3,2024/02/03,2024-02-06,SKU-SP001,Bluetooth-högtalare,Ljud,1,899.00,Göteborg,Privat,Kort,KND-38153,Levererad,NaN,NaN,NaN


In [38]:
df_raw["region"].value_counts()

region
Stockholm     834
Göteborg      434
Malmö         236
Uppsala       192
Norrland      125
Örebro        117
Linköping     117
Västerås       75
STOCKHOLM      50
Sthml          44
stockholm      42
STHLM          39
uppsala        25
Sthlm          25
göteborg       24
GÖTEBORG       22
UPPSALA        21
Gothenburg     19
MALMÖ          16
Gbg            14
malmo          12
LINKÖPING      11
GBGB           11
Orebro         10
Vasteras       10
örebro          9
ÖREBRO          9
norrland        9
linköping       8
NORRLAND        8
Malmo           8
Linkoping       8
västerås        7
Norr            7
VÄSTERÅS        7
malmö           7
Name: count, dtype: int64

In [39]:
df_raw["kundtyp"].value_counts()

kundtyp
Privat       1535
Företag       783
privat         66
b2c            64
Konsument      60
PRIVAT         58
B2C            45
B2B            39
b2b            35
FÖRETAG        29
Firma          27
företag        26
Name: count, dtype: int64

In [40]:
df_raw["betalmetod"].value_counts()

betalmetod
Faktura           938
Kort              775
Swish             550
Invoice            51
FAKTURA            50
faktura            44
Kreditkort         36
KORT               33
SWISH              31
swish              31
kort               30
Mobilbetalning     29
Visa               28
Mastercard         25
Name: count, dtype: int64

In [41]:
df_raw["leveransstatus"].value_counts()

leveransstatus
Levererad          2005
Under transport     152
Retur               132
Skickad              95
Mottagen             92
levererad            86
LEVERERAD            68
Returnerad           10
Återsänd              8
På väg                7
retur                 6
under transport       6
RETUR                 4
UNDER TRANSPORT       2
Name: count, dtype: int64

In [25]:
df_raw["orderrad_id"].isna().sum()


np.int64(0)

In [ ]:
df_raw.duplicated().sum()

np.int64(67)

In [27]:
df_raw["orderrad_id"].duplicated().sum()


np.int64(67)

In [31]:
df_raw["order_id"].duplicated().sum()

np.int64(1110)

In [29]:
dupes = df_raw[df_raw["orderrad_id"].duplicated(keep=False)].sort_values("orderrad_id")
dupes.head(10)


,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
613,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
107,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
181,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
1403,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
2003,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
182,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
224,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
1682,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
276,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0
383,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0


In [30]:
(df_raw[["order_id", "orderrad_id"]]
 .duplicated()
 .sum())


np.int64(67)

In [32]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         2767 non-null   str    
 1   orderrad_id      2767 non-null   str    
 2   orderdatum       2767 non-null   str    
 3   leveransdatum    2767 non-null   str    
 4   produkt_sku      2767 non-null   str    
 5   produktnamn      2767 non-null   str    
 6   kategori         2767 non-null   str    
 7   antal            2767 non-null   str    
 8   pris_per_enhet   2767 non-null   str    
 9   region           2612 non-null   str    
 10  kundtyp          2767 non-null   str    
 11  betalmetod       2651 non-null   str    
 12  kund_id          2767 non-null   str    
 13  leveransstatus   2673 non-null   str    
 14  recension_text   1355 non-null   str    
 15  recensionsdatum  1355 non-null   str    
 16  betyg            1355 non-null   float64
dtypes: float64(1), str(16)
me

# EDA

1. Orderrad_id är unikt och är datasetets grain, men det finns dubletter som behöver hanteras.

2. Orderdatum är en sträng som behöver göras om till datetime, samma gäller för leveransdatum. 

3. Antal ska ändras om till en int. 

4. Pris_per_enhet ska göras om till float. 

5. Region beöver standardiseras och nullvärden hanteras. 

6. Kundtyp behöver standardiseras till rätt namn. 

7. betalmetod behöver standardiseras till rätt namn

8. Leveransstatus behöver standardiseras till rätt namn.

9. Behöver göra om recensionsdatum till datetime. 

In [ ]:
#1. fixa dubletter i orderrader
def clean_orderrader(df):
    df_clean = df.copy()
    # Ta bort dubbletter baserat på 'orderrad_id'
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    after = len(df_clean)

    print(f"Tagit bort dubletter: {before - after}")

    return df_clean



Tagit bort dubletter: 67


In [ ]:
#2. fixa orderdatum
def clean_orderdatum(df):
    df_clean = df.copy()
    
    # Konvertera "orderdatum" till datetime-format
    df_clean['orderdatum'] = pd.to_datetime(
        df_clean['orderdatum'], 
        dayfirst=True, 
        format="mixed", 
        errors='coerce')

    # Här hanterar vi ogiltiga datum genom att sätta dem till NaT
    invalid_dates = df_clean['orderdatum'].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'orderdatum', sätter dessa till NaT.")

    return df_clean

In [ ]:
#2.1 fixa leveransdatum
def clean_leveransdatum(df):
    df_clean = df.copy()
    
    # Konvertera 'leveransdatum' till datetime-format
    df_clean['leveransdatum'] = pd.to_datetime(
        df_clean['leveransdatum'], 
        dayfirst=True, 
        format="mixed", 
        errors='coerce')

    # Här hanterar vi ogiltiga datum genom att sätta dem till NaT
    invalid_dates = df_clean['leveransdatum'].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'leveransdatum', sätter dessa till NaT.")

        

    return df_clean

In [48]:
#3.
def clean_antal(df):
    df_clean = df.copy()
    
    # Konvertera 'antal' till numeriskt format
    df_clean['antal'] = pd.to_numeric(
        df_clean['antal'], 
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_antal = (df_clean['antal'] <= 0).sum()
    if invalid_antal > 0:
        print(f"Hittade {invalid_antal} ogiltiga värden i 'antal' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['antal'] <= 0, 'antal'] = np.nan

    df_clean["antal"] = df_clean["antal"].astype("Int64")

    return df_clean

In [64]:
#4.
def clean_pris(df):
    df_clean = df.copy()
    
    # Konvertera 'pris' till numeriskt format
    df_clean['pris'] = pd.to_numeric(
        df_clean['pris'], 
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_pris = (df_clean['pris'] <= 0).sum()
    if invalid_pris > 0:
        print(f"Hittade {invalid_pris} ogiltiga värden i 'pris' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['pris'] <= 0, 'pris'] = np.nan

    return df_clean

In [63]:
#5.
def standardisera_region(df):
    df_clean = df.copy()
    df_clean["region_raw"] = df_clean["region"]

    def klassificera_region(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        if v in ["stockholm", "sthlm", "sthl", "sthml"]:
            return "Stockholm"

        if v in ["uppsala"]:
            return "Uppsala"

        if v in ["göteborg", "gothenburg", "gbg", "gbgb"]:
            return "Göteborg"

        if v in ["malmö", "malmo"]:
            return "Malmö"

        if v in ["norrland", "norr"]:
            return "Norrland"

        if v in ["örebro", "orebro"]:
            return "Örebro"

        if v in ["västerås", "vasteras"]:
            return "Västerås"

        if v in ["linköping", "linkoping"]:
            return "Linköping"

        return "Okänd"

    df_clean["region"] = df_clean["region"].apply(klassificera_region)
    return df_clean


In [ ]:
#7.

In [ ]:
#8.

In [ ]:
#9.